In [ ]:
! pip install unsloth

In [ ]:
from unsloth import FastModel
import torch

In [ ]:
model, tokenizer = FastModel.from_pretrained(
    model_name = "sarvamai/sarvam-translate",
    max_seq_length = 1024,
    load_in_4bit = True,
)

In [ ]:
model.eval()

In [ ]:
import pandas as pd
import torch
from tqdm import tqdm

# -----------------------------
# Config
# -----------------------------
CSV_FILE = "aligned_telugu_eng_output.csv"
SOURCE_COL = "English_Reference"
TARGET_COL = "Model_Telugu"
TGT_LANG = "Telugu"

MAX_INPUT_LEN = 256
MAX_NEW_TOKENS = 256

# -----------------------------
# Load CSV
# -----------------------------
df = pd.read_csv(CSV_FILE, encoding="utf-8", on_bad_lines="skip")
df = df.head(10)   # debug

print("Columns:", df.columns.tolist())

sentences = df[SOURCE_COL].astype(str).tolist()

print("\nSample inputs:")
for s in sentences[:3]:
    print(">>>", s)

translations = []

# -----------------------------
# Translation Loop (chat template)
# -----------------------------
for sent in tqdm(sentences):

    # Safety clip for long text
    sent = sent[:800]

    messages = [
        {"role": "system", "content": f"You are a professional translator. Translate the text to {TGT_LANG}."},
        {"role": "user", "content": sent},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        text=prompt,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LEN,
    ).to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,       # deterministic
            temperature=0.0,
        )

    # Remove prompt tokens
    output_ids = generated_ids[0][inputs.input_ids.shape[1]:]

    translation = tokenizer.decode(
        output_ids,
        skip_special_tokens=True
    ).strip()

    translations.append(translation)

# -----------------------------
# Save Results
# -----------------------------
df[TARGET_COL] = translations
OUT_FILE = "aligned_telugu_with_gemma3_chat_template.csv"
df.to_csv(OUT_FILE, index=False, encoding="utf-8")

print(f"\n✅ Saved to: {OUT_FILE}")

print("\nSample outputs:")
for t in translations[:3]:
    print("<<<", t)


In [ ]:
df.head()

In [ ]:
df["telugu_translation"] = translations
df.to_csv("aligned_labse_and_sarvam.csv", index=False)

print("aligned_labse_and_sarvam.csv")


In [ ]:
!pip install -q sacrebleu pandas


In [ ]:
import pandas as pd
import sacrebleu

# -----------------------------
# Config
# -----------------------------
CSV_FILE = "aligned_labse_and_sarvam.csv"

REF_COL = "Telugu_Reference"      # CHANGE to your true Telugu reference column
HYP_COL = "telugu_translation"   # or "Model_Telugu"

# -----------------------------
# Load CSV
# -----------------------------
df = pd.read_csv(CSV_FILE, encoding="utf-8", on_bad_lines="skip")

refs = df[REF_COL].astype(str).tolist()
hyps = df[HYP_COL].astype(str).tolist()

assert len(refs) == len(hyps)

# -----------------------------
# BLEU
# -----------------------------
bleu = sacrebleu.corpus_bleu(hyps, [refs])

# -----------------------------
# chrF++
# -----------------------------
chrf = sacrebleu.corpus_chrf(hyps, [refs])

print("========== MT Evaluation ==========")
print(f"BLEU   : {bleu.score:.2f}")
print(f"chrF++ : {chrf.score:.2f}")
print("===================================")
